# Image SEO + multimodal audit (MLforSEO)

Companion notebook for Lazarina Stoy's MLforSEO guide on **image SEO, advanced tactics and multimodal search**.

What it does, for a list of your pages:
1. finds every `<img>`, `<picture>` source, inline CSS background image and `og:image`;
2. audits each image: real format, file size, pixel size, alt text, file name, width/height attributes, `srcset`, lazy-loading on the first image, AVIF/WebP negotiation;
3. **optionally** drafts alt text for images with no alt attribute, using an open vision-language model (runs in Colab, no key) or the OpenAI / Gemini API (key from Colab secrets, never hardcoded);
4. **optionally** finds near-duplicate images with CLIP embeddings plus a perceptual hash.

Suggested alt text is a draft for a human to check. The guide explains why.

**Runtime:** CPU works (about 5-10 s per image for alt text with SmolVLM-500M). For Qwen3-VL-2B, switch to a T4 GPU (Runtime > Change runtime type).

In [ ]:
# 1. Install and download the script
!pip install -q requests beautifulsoup4 lxml pandas "pillow>=11.3" ImageHash transformers accelerate torchvision
!wget -q -O image_seo_audit.py https://raw.githubusercontent.com/lazarinastoy/mlforseo-code/main/image-seo-multimodal/image_seo_audit.py
import image_seo_audit as audit
import pandas as pd
pd.set_option("display.max_colwidth", 80)

## 2. Your pages

One URL per line. Start with 5-20 important templates (home, category, product, article) rather than the whole site: the script downloads every image once and is polite (1 s between pages).

In [ ]:
urls = """
https://www.mlforseo.com/
https://developers.google.com/search/docs/appearance/google-images
https://web.dev/articles/image-cdns
""".split()

## 3. Choose your options

* `MAX_KB`: flag images heavier than this. There is no official Google number; 100-200 KB is a common audit threshold (Screaming Frog's default is 100 KB).
* `ALT_MAX`: flag alt text longer than this. Not a Google limit, it's an accessibility convention: long alt text is tiring in a screen reader.
* `CLASSIFY` / `LABELS`: CLIP zero-shot classification. Write labels as short captions ("a product photo on a plain white background"); the model picks the closest one for each image.
* `ALT_BACKEND`: `"hf"` (open model, no key), `"openai"` or `"gemini"`. For the APIs, add `OPENAI_API_KEY` or `GEMINI_API_KEY` in Colab's Secrets panel (key icon on the left).

In [ ]:
MAX_KB = 200
ALT_MAX = 125
SUGGEST_ALT = True
ALT_BACKEND = "hf"          # "hf" | "openai" | "gemini"
ALT_MODEL = None            # e.g. "Qwen/Qwen3-VL-2B-Instruct" on a GPU runtime
DEDUPE = True
CLASSIFY = True           # zero-shot CLIP labels (edit LABELS to suit your site)
LABELS = audit.DEFAULT_LABELS

import os
if ALT_BACKEND in ("openai", "gemini"):
    from google.colab import userdata
    key = "OPENAI_API_KEY" if ALT_BACKEND == "openai" else "GEMINI_API_KEY"
    os.environ[key] = userdata.get(key)

In [ ]:
# 4. Run the audit
res = audit.run(urls, out="results", max_kb=MAX_KB, alt_max=ALT_MAX,
                suggest_alt=SUGGEST_ALT, alt_backend=ALT_BACKEND, alt_model=ALT_MODEL,
                alt_limit=25, dedupe=DEDUPE,
                classify=CLASSIFY, labels=LABELS)

## 5. Page summary

In [ ]:
res['pages']

## 6. Issues across all images

In [ ]:
images = res['images']
images['issues'].str.split('; ').explode().value_counts()

In [ ]:
# Every image with at least one issue
images[images.issues != ''][['page_url', 'image_url', 'format', 'bytes', 'px_width', 'alt', 'issues']]

## 6b. Your image mix (CLIP zero-shot labels)


In [ ]:
if 'clip_label' in images:
    display(images.drop_duplicates('image_url')['clip_label'].value_counts())


## 7. Alt-text drafts

Review every one of these before it goes anywhere near your CMS. Check facts the model can't know (names, products, places, what a chart *means*), and rewrite for the page's purpose.

In [ ]:
images[images.alt_suggestion != ''][['page_url', 'image_url', 'figcaption', 'alt_suggestion']]

## 8. Near-duplicates

* `phash_distance` <= 8: the same picture, resized or re-encoded. Pick one canonical URL and reuse it.
* High `clip_cosine` with a large pHash distance: visually near-identical (same template, scene or crop). Decide whether you need both.

In [ ]:
res.get('duplicates', pd.DataFrame())

In [ ]:
# 9. Download the CSVs
from google.colab import files
for f in ['images.csv', 'pages.csv', 'duplicates.csv']:
    if os.path.exists(f'results/{f}'):
        files.download(f'results/{f}')